# 실습 - 깊이 추정

**단일 이미지에서 원근 추정**

**작업 개요**

- 작업 이름 — `depth-estimation`
- 입력 — 이미지 한 장
- 출력 — 깊이 지도(이미지 형태) · 수치 배열
- 밝을수록 가까움 · 어두울수록 멀리
- **상대 깊이** — 단위 없음, 절대 거리 아님

**앞선 작업과의 차이**

| 구분 | 출력 |
|---|---|
| 분류 | 라벨 · 점수 |
| 탐지 | 라벨 · 점수 · 상자 |
| 세그멘테이션 | 마스크 |
| **깊이 추정** | **픽셀마다 깊이 값** |

**데이터**

- 실내외 일상 장면 이미지
- 150개 범주 — 하늘 · 도로 · 사람 · 가구 등
- 일반 장면 — 학습 데이터와 성격 유사

**모델 — DPT (Dense Prediction Transformer)**

- 카메라 한 대로 원근을 추정 — 별도 장비 불필요
- 3차원 형상 · 표면 요철 파악에 활용

**실습 순서**

- 환경 준비
- 데이터 불러오기
- 모델 준비 · 추론 실행
- 원본과 나란히 비교 · 수치 확인
- 전문 영상에 적용 — 한계 확인

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행
- `PIL` · `numpy` — 이미지 배치와 수치 확인

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from PIL import Image
import numpy as np
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 실내외 일상 장면 이미지
- `split="train[:50]"` — 앞 50장만 사용
- ⑤ 이미지 설명 생성과 **동일 데이터** 사용

In [ ]:
ds = load_dataset(
    "merve/scene_parse_150",
    split="train[:50]"
)

ds

### 1-2. 이미지 확인

- 원근이 뚜렷한 장면일수록 결과가 잘 드러남
- 실내 장면 — 가구·벽까지의 거리 차이
- 실외 장면 — 도로·건물·하늘의 거리 차이

In [ ]:
image = ds[0]["image"]

print("이미지 크기:", image.size)
image

---
## 2. 모델 준비

- `task="depth-estimation"` — 깊이 추정 작업
- `model` — DPT (Dense Prediction Transformer)
- `device=0` — GPU에서 실행

첫 실행 시 모델 내려받기로 시간이 소요된다.

In [ ]:
depth = pipeline(
    task="depth-estimation",
    model="Intel/dpt-hybrid-midas",
    device=0
)

print("모델 준비 완료")

---
## 3. 추론 실행

- 출력은 두 가지 항목으로 구성
- `depth` — 화면 표시용 이미지 (흑백)
- `predicted_depth` — 수치 배열 (계산용)

In [ ]:
out = depth(image)

print("출력 항목:", list(out.keys()))
out["depth"]

### 3-1. 결과 읽기

- **밝은 영역** — 카메라에 가까움
- **어두운 영역** — 멀리 있음
- 원본과 나란히 놓으면 대응 관계가 분명해짐

In [ ]:
def side_by_side(img_left, img_right):
    """두 이미지를 좌우로 나란히 배치"""
    l = img_left.convert("RGB")
    r = img_right.convert("RGB").resize(l.size)

    w, h = l.size
    pair = Image.new("RGB", (w * 2, h), "white")
    pair.paste(l, (0, 0))
    pair.paste(r, (w, 0))
    return pair


side_by_side(image, out["depth"])

### 3-2. 수치 확인

- `predicted_depth` — 픽셀마다 하나의 깊이 값
- 값의 범위는 이미지마다 상이 — **상대 깊이**
- 단위가 없으므로 "몇 미터"로 해석할 수 없음

In [ ]:
d = out["predicted_depth"]

if hasattr(d, "cpu"):
    d = d.cpu().numpy()
d = np.asarray(d)

print("형태:", d.shape)
print("최솟값:", round(float(d.min()), 2))
print("최댓값:", round(float(d.max()), 2))
print("평균:", round(float(d.mean()), 2))

### 3-3. 값의 의미 확인

- 깊이 지도의 특정 위치 값을 직접 확인
- 가까운 대상과 먼 대상의 값 차이 비교

In [ ]:
h, w = d.shape[-2:]
arr = d.reshape(h, w)

print("중앙 값 :", round(float(arr[h // 2, w // 2]), 2))
print("좌상단  :", round(float(arr[h // 10, w // 10]), 2))
print("좌하단  :", round(float(arr[-h // 10, w // 10]), 2))
print()
print("값이 클수록 가까움")

---
## 4. 여러 이미지 확인

- 장면에 따라 결과가 어떻게 달라지는지 관찰
- 원근이 뚜렷한 장면 · 평면적인 장면 비교

In [ ]:
for n in [1, 2, 3]:
    img = ds[n]["image"]
    res = depth(img)
    display(side_by_side(img, res["depth"]))

---
## 5. 전문 영상에 적용

- 앞선 실습에서 사용한 이미지에 동일 모델 적용
- 학습 데이터(일상 장면)와 성격이 다른 경우 결과 확인

**예상**

- 현미경 · 기판 영상은 원근 개념이 약함
- 결과가 나오더라도 물리적 의미로 해석하기 어려움

In [ ]:
# 앞선 실습 데이터 — PCB 기판
ds_pcb = load_dataset(
    "keremberke/pcb-defect-segmentation",
    name="full", split="train[:5]"
)
pcb = ds_pcb[0]["image"]

res_pcb = depth(pcb)
side_by_side(pcb, res_pcb["depth"])

**확인 사항**

- 결과 이미지는 생성되지만 원근 정보로 보기 어려움
- 모델은 항상 결과를 산출 — 의미 있는 결과라는 뜻은 아님
- 학습 데이터의 성격이 적용 범위를 결정

---
## 6. 다른 이미지 확인

- 이미지 번호를 바꾸어 다른 장면 확인
- 본인 분야 이미지에 적용 — 원근이 있는 사진일수록 잘 동작
- 웹 이미지 주소로도 실행 가능 — `depth("이미지 주소")`

In [ ]:
n = 10
img = ds[n]["image"]

res = depth(img)
side_by_side(img, res["depth"])

---
## 7. 정리

**수행 내용**

- 사진 한 장에서 원근을 추정
- 깊이 지도(이미지)와 수치 배열을 동시에 획득
- 원본과 나란히 비교하여 대응 관계 확인

**결과 해석**

- 밝을수록 가까움 · 어두울수록 멀리
- **상대 깊이** — 단위 없음, 절대 거리로 해석 불가
- 값의 범위는 이미지마다 상이

**적용 범위**

- 일반 장면 — 학습 데이터와 성격 유사, 양호
- 전문 영상(현미경 · 기판) — 결과는 나오나 의미 해석 곤란
- 학습 데이터의 성격이 적용 범위를 결정

**활용**

- 3차원 형상 · 표면 요철 파악
- 카메라 한 대로 원근 추정 — 별도 장비 불필요

---

데이터 출처: merve/scene_parse_150 (Hugging Face Hub 공개)